# 🔤 Task 5: Fine-tune DistilBERT for POS Tagging & Chunking

**Internship:** Data Science Internship – February 2026  
**Task:** Fine-tune a transformer model (BERT/DistilBERT) to perform:  
- Part-of-Speech (POS) Tagging  
- Chunking (Shallow Parsing)  

**Model Used:** `distilbert-base-uncased`  
**Dataset:** CoNLL-2003 (via HuggingFace Datasets)  

---

## 📌 Table of Contents
1. Introduction & Theory
2. Environment Setup & Installations
3. Dataset Loading & Exploration
4. Data Preprocessing & Tokenization
5. Model Architecture
6. Fine-tuning for POS Tagging
7. Fine-tuning for Chunking
8. Evaluation & Metrics
9. Inference on Custom Sentences
10. Results Visualization
11. Conclusion

## 1. 📚 Introduction & Theory

### What is POS Tagging?
**Part-of-Speech (POS) Tagging** is the process of assigning grammatical labels (noun, verb, adjective, etc.) to each word in a sentence.

Example:  
`The (DT) cat (NN) sat (VBD) on (IN) the (DT) mat (NN)`

### What is Chunking?
**Chunking** (Shallow Parsing) groups words into meaningful "chunks" based on POS patterns, such as Noun Phrases (NP) or Verb Phrases (VP). It uses BIO (Beginning, Inside, Outside) tagging:
- **B-NP**: Beginning of a Noun Phrase
- **I-NP**: Inside a Noun Phrase
- **O**: Outside any chunk

### Why DistilBERT?
DistilBERT is a lighter, faster version of BERT that retains ~97% of BERT's performance while being 40% smaller and 60% faster — ideal for fine-tuning NLP tasks.

### Architecture for Token Classification
We add a **linear classification head** on top of DistilBERT's token embeddings to predict a label for each token.

## 2. ⚙️ Environment Setup & Installations

In [ ]:
# Install required libraries
# Run this cell once in Google Colab or your local environment
!pip install transformers datasets seqeval accelerate evaluate -q

In [ ]:
# ─────────────────────────────────────────
# Core Imports
# ─────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# HuggingFace Libraries
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate
import torch

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")
print(f"✅ PyTorch version: {torch.__version__}")

## 3. 📊 Dataset Loading & Exploration

We use the **CoNLL-2003** dataset — the standard benchmark for NLP sequence labeling tasks. It contains:
- **POS tags** (45 Penn Treebank tags)
- **Chunk tags** (BIO format for NP, VP, PP, etc.)
- **NER tags** (we won't use these)

In [ ]:
# ─────────────────────────────────────────
# Load the CoNLL-2003 Dataset
# ─────────────────────────────────────────
print("📥 Loading CoNLL-2003 dataset...")
raw_datasets = load_dataset("conll2003", trust_remote_code=True)

print("\n📋 Dataset Structure:")
print(raw_datasets)

print("\n📋 Features:")
print(raw_datasets["train"].features)

In [ ]:
# ─────────────────────────────────────────
# Explore a sample from the dataset
# ─────────────────────────────────────────
sample = raw_datasets["train"][0]

# Extract label names
pos_label_names = raw_datasets["train"].features["pos_tags"].feature.names
chunk_label_names = raw_datasets["train"].features["chunk_tags"].feature.names

print(f"\n🔤 Sample Tokens:  {sample['tokens']}")
print(f"\n🏷️  POS Tags (IDs): {sample['pos_tags']}")
print(f"   POS Tags (Names): {[pos_label_names[i] for i in sample['pos_tags']]}")
print(f"\n📦 Chunk Tags (IDs):   {sample['chunk_tags']}")
print(f"   Chunk Tags (Names): {[chunk_label_names[i] for i in sample['chunk_tags']]}")

print(f"\n📌 Total POS Labels:   {len(pos_label_names)} → {pos_label_names}")
print(f"📌 Total Chunk Labels: {len(chunk_label_names)} → {chunk_label_names}")

In [ ]:
# ─────────────────────────────────────────
# Visualize POS Tag Distribution
# ─────────────────────────────────────────
all_pos_ids = [tag for ex in raw_datasets["train"] for tag in ex["pos_tags"]]
pos_counts = Counter(all_pos_ids)
top_pos = sorted(pos_counts.items(), key=lambda x: x[1], reverse=True)[:20]

labels_top = [pos_label_names[t[0]] for t in top_pos]
counts_top = [t[1] for t in top_pos]

plt.figure(figsize=(14, 5))
bars = plt.bar(labels_top, counts_top, color=plt.cm.tab20.colors[:20])
plt.title("Top 20 POS Tags in CoNLL-2003 Training Set", fontsize=14, fontweight='bold')
plt.xlabel("POS Tag")
plt.ylabel("Count")
plt.xticks(rotation=45, ha='right')
for bar, count in zip(bars, counts_top):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
             str(count), ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.savefig("pos_distribution.png", dpi=150)
plt.show()
print("✅ POS distribution chart saved.")

In [ ]:
# ─────────────────────────────────────────
# Visualize Chunk Tag Distribution
# ─────────────────────────────────────────
all_chunk_ids = [tag for ex in raw_datasets["train"] for tag in ex["chunk_tags"]]
chunk_counts = Counter(all_chunk_ids)

chunk_labels_sorted = sorted(chunk_counts.items(), key=lambda x: x[1], reverse=True)
c_labels = [chunk_label_names[t[0]] for t in chunk_labels_sorted]
c_counts = [t[1] for t in chunk_labels_sorted]

plt.figure(figsize=(14, 5))
bars = plt.bar(c_labels, c_counts, color=plt.cm.Set2.colors[:len(c_labels)])
plt.title("Chunk Tag Distribution in CoNLL-2003 Training Set", fontsize=14, fontweight='bold')
plt.xlabel("Chunk Tag")
plt.ylabel("Count")
plt.xticks(rotation=45, ha='right')
for bar, count in zip(bars, c_counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
             str(count), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig("chunk_distribution.png", dpi=150)
plt.show()
print("✅ Chunk distribution chart saved.")

## 4. 🔧 Data Preprocessing & Tokenization

DistilBERT uses **WordPiece tokenization**, which may split a word into multiple subword tokens. We need to carefully **align the labels** with the tokenized output:
- For the **first subword** of a word → assign the word's label
- For **subsequent subwords** → assign `-100` (ignored in loss computation)

In [ ]:
# ─────────────────────────────────────────
# Initialize Tokenizer
# ─────────────────────────────────────────
MODEL_CHECKPOINT = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(f"✅ Tokenizer loaded: {MODEL_CHECKPOINT}")

# Demonstrate tokenization on a sample sentence
sample_sentence = ["Apple", "is", "looking", "at", "buying", "U.K.", "startup"]
tokenized = tokenizer(sample_sentence, is_split_into_words=True)
print(f"\n🔍 Original words:    {sample_sentence}")
print(f"🔍 Tokenized input:   {tokenizer.convert_ids_to_tokens(tokenized['input_ids'])}")
print(f"🔍 Word IDs:          {tokenized.word_ids()}")
print("   (None = [CLS]/[SEP], integers show which original word each subtoken belongs to)")

In [ ]:
# ─────────────────────────────────────────
# Label Alignment Function
# ─────────────────────────────────────────
def tokenize_and_align_labels(examples, label_column):
    """
    Tokenizes examples and aligns labels with subword tokens.
    
    Strategy:
    - First subword of a word: assign the word's true label
    - Subsequent subwords: assign -100 (ignored in CrossEntropyLoss)
    - Special tokens ([CLS], [SEP]): assign -100
    """
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128
    )

    all_labels = []
    for i, label in enumerate(examples[label_column]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                # Special token → ignore
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # First subword of a new word → assign real label
                label_ids.append(label[word_idx])
            else:
                # Continuation subword → ignore
                label_ids.append(-100)

            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs


print("✅ Label alignment function defined.")

# Quick test
test_batch = raw_datasets["train"].select([0])
aligned = tokenize_and_align_labels(test_batch, "pos_tags")
print(f"\n🔍 Sample aligned labels: {aligned['labels'][0][:15]}")
print("   (-100 = special/continuation token, integers = label IDs)")

## 5. 🏷️ Fine-tuning DistilBERT for POS Tagging

In [ ]:
# ─────────────────────────────────────────
# Tokenize Dataset for POS Tagging
# ─────────────────────────────────────────
print("🔄 Tokenizing dataset for POS Tagging...")

pos_tokenized = raw_datasets.map(
    lambda x: tokenize_and_align_labels(x, "pos_tags"),
    batched=True,
    remove_columns=raw_datasets["train"].column_names
)

print("✅ Tokenization complete!")
print(f"   Train size:      {len(pos_tokenized['train'])}")
print(f"   Validation size: {len(pos_tokenized['validation'])}")
print(f"   Test size:       {len(pos_tokenized['test'])}")

In [ ]:
# ─────────────────────────────────────────
# Build Label-to-ID Mappings for POS
# ─────────────────────────────────────────
pos_id2label = {i: label for i, label in enumerate(pos_label_names)}
pos_label2id = {label: i for i, label in enumerate(pos_label_names)}

print(f"📌 Number of POS labels: {len(pos_label_names)}")
print(f"   Sample mappings: {dict(list(pos_id2label.items())[:5])}")

In [ ]:
# ─────────────────────────────────────────
# Initialize DistilBERT for POS Tagging
# ─────────────────────────────────────────
pos_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(pos_label_names),
    id2label=pos_id2label,
    label2id=pos_label2id
)

total_params = sum(p.numel() for p in pos_model.parameters())
trainable_params = sum(p.numel() for p in pos_model.parameters() if p.requires_grad)
print(f"✅ POS Model loaded.")
print(f"   Total parameters:     {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Classification head:  DistilBERT hidden (768) → {len(pos_label_names)} POS labels")

In [ ]:
# ─────────────────────────────────────────
# Define Metrics (seqeval F1 score)
# ─────────────────────────────────────────
seqeval = evaluate.load("seqeval")

def compute_metrics_pos(eval_preds):
    """Compute token-level accuracy and seqeval F1 for POS."""
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    # Remove -100 (ignored tokens) and map IDs to label strings
    true_labels = [
        [pos_label_names[l] for l in label if l != -100]
        for label in labels
    ]
    true_predictions = [
        [pos_label_names[p] for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )
    return {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results["overall_accuracy"],
    }

print("✅ Metrics function defined (Precision, Recall, F1, Accuracy).")

In [ ]:
# ─────────────────────────────────────────
# Training Arguments for POS Tagging
# ─────────────────────────────────────────
pos_training_args = TrainingArguments(
    output_dir="./pos_tagger",             # Save directory
    evaluation_strategy="epoch",           # Evaluate after each epoch
    save_strategy="epoch",                 # Save checkpoint each epoch
    learning_rate=2e-5,                    # Standard LR for fine-tuning BERT models
    per_device_train_batch_size=16,        # Batch size per GPU/CPU
    per_device_eval_batch_size=16,
    num_train_epochs=3,                    # Fine-tune for 3 epochs
    weight_decay=0.01,                     # L2 regularization
    warmup_ratio=0.1,                      # Linear warmup for 10% of steps
    load_best_model_at_end=True,           # Load best checkpoint after training
    metric_for_best_model="f1",            # Choose best model by F1
    logging_dir="./logs_pos",              # TensorBoard logs
    logging_steps=100,                     # Log every 100 steps
    report_to="none",                      # Disable W&B / cloud reporting
    fp16=torch.cuda.is_available(),        # Mixed precision if GPU available
)

print("✅ POS Training arguments configured.")
print(f"   Epochs: {pos_training_args.num_train_epochs}")
print(f"   LR:     {pos_training_args.learning_rate}")
print(f"   Batch:  {pos_training_args.per_device_train_batch_size}")

In [ ]:
# ─────────────────────────────────────────
# Initialize Trainer for POS Tagging
# ─────────────────────────────────────────
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

pos_trainer = Trainer(
    model=pos_model,
    args=pos_training_args,
    train_dataset=pos_tokenized["train"],
    eval_dataset=pos_tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_pos
)

print("✅ POS Trainer initialized.")

In [ ]:
# ─────────────────────────────────────────
# 🚀 Train the POS Tagger
# ─────────────────────────────────────────
print("🚀 Starting POS Tagging fine-tuning...")
print("   This may take ~10-20 minutes on GPU, or longer on CPU.")
print("-" * 60)

pos_train_result = pos_trainer.train()

print("\n✅ POS Training Complete!")
print(f"   Training Loss:  {pos_train_result.training_loss:.4f}")
print(f"   Total Steps:    {pos_train_result.global_step}")
print(f"   Time (seconds): {pos_train_result.metrics['train_runtime']:.1f}")

In [ ]:
# ─────────────────────────────────────────
# Evaluate POS Tagger on Test Set
# ─────────────────────────────────────────
print("📊 Evaluating POS Tagger on Test Set...")

pos_test_results = pos_trainer.evaluate(pos_tokenized["test"])

print("\n📋 POS Tagging Test Results:")
print(f"   Precision: {pos_test_results['eval_precision']:.4f}")
print(f"   Recall:    {pos_test_results['eval_recall']:.4f}")
print(f"   F1 Score:  {pos_test_results['eval_f1']:.4f}")
print(f"   Accuracy:  {pos_test_results['eval_accuracy']:.4f}")
print(f"   Eval Loss: {pos_test_results['eval_loss']:.4f}")

In [ ]:
# Save the POS model
pos_model.save_pretrained("./pos_tagger_final")
tokenizer.save_pretrained("./pos_tagger_final")
print("✅ POS model saved to ./pos_tagger_final")

## 6. 📦 Fine-tuning DistilBERT for Chunking

We now train a **separate model** on Chunk tags from CoNLL-2003. Chunk tags follow BIO format (e.g., B-NP, I-NP, B-VP, O).

In [ ]:
# ─────────────────────────────────────────
# Tokenize Dataset for Chunking
# ─────────────────────────────────────────
print("🔄 Tokenizing dataset for Chunking...")

chunk_tokenized = raw_datasets.map(
    lambda x: tokenize_and_align_labels(x, "chunk_tags"),
    batched=True,
    remove_columns=raw_datasets["train"].column_names
)

print("✅ Chunk tokenization complete!")

# Label mappings for chunking
chunk_id2label = {i: label for i, label in enumerate(chunk_label_names)}
chunk_label2id = {label: i for i, label in enumerate(chunk_label_names)}

print(f"\n📌 Chunk labels ({len(chunk_label_names)}): {chunk_label_names}")

In [ ]:
# ─────────────────────────────────────────
# Initialize DistilBERT for Chunking
# ─────────────────────────────────────────
chunk_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(chunk_label_names),
    id2label=chunk_id2label,
    label2id=chunk_label2id
)

print(f"✅ Chunking Model loaded.")
print(f"   Classification head: DistilBERT hidden (768) → {len(chunk_label_names)} Chunk labels")

In [ ]:
# ─────────────────────────────────────────
# Metrics for Chunking
# ─────────────────────────────────────────
def compute_metrics_chunk(eval_preds):
    """Compute seqeval metrics for Chunking."""
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_labels = [
        [chunk_label_names[l] for l in label if l != -100]
        for label in labels
    ]
    true_predictions = [
        [chunk_label_names[p] for p, l in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )
    return {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results["overall_accuracy"],
    }

print("✅ Chunking metrics function defined.")

In [ ]:
# ─────────────────────────────────────────
# Training Arguments & Trainer for Chunking
# ─────────────────────────────────────────
chunk_training_args = TrainingArguments(
    output_dir="./chunker",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs_chunk",
    logging_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

chunk_trainer = Trainer(
    model=chunk_model,
    args=chunk_training_args,
    train_dataset=chunk_tokenized["train"],
    eval_dataset=chunk_tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_chunk
)

print("✅ Chunking Trainer initialized.")

In [ ]:
# ─────────────────────────────────────────
# 🚀 Train the Chunker
# ─────────────────────────────────────────
print("🚀 Starting Chunking fine-tuning...")
print("-" * 60)

chunk_train_result = chunk_trainer.train()

print("\n✅ Chunking Training Complete!")
print(f"   Training Loss:  {chunk_train_result.training_loss:.4f}")
print(f"   Total Steps:    {chunk_train_result.global_step}")
print(f"   Time (seconds): {chunk_train_result.metrics['train_runtime']:.1f}")

In [ ]:
# ─────────────────────────────────────────
# Evaluate Chunker on Test Set
# ─────────────────────────────────────────
print("📊 Evaluating Chunker on Test Set...")

chunk_test_results = chunk_trainer.evaluate(chunk_tokenized["test"])

print("\n📋 Chunking Test Results:")
print(f"   Precision: {chunk_test_results['eval_precision']:.4f}")
print(f"   Recall:    {chunk_test_results['eval_recall']:.4f}")
print(f"   F1 Score:  {chunk_test_results['eval_f1']:.4f}")
print(f"   Accuracy:  {chunk_test_results['eval_accuracy']:.4f}")
print(f"   Eval Loss: {chunk_test_results['eval_loss']:.4f}")

In [ ]:
# Save the Chunking model
chunk_model.save_pretrained("./chunker_final")
tokenizer.save_pretrained("./chunker_final")
print("✅ Chunking model saved to ./chunker_final")

## 7. 🔮 Inference on Custom Sentences

Now let's run both models on new, unseen sentences and interpret the predictions.

In [ ]:
# ─────────────────────────────────────────
# Inference Helper Function
# ─────────────────────────────────────────
def predict_tags(sentence: str, model, tokenizer, id2label: dict) -> list:
    """
    Predicts tags for each word in a sentence.
    
    Args:
        sentence: Raw string sentence
        model: Fine-tuned token classification model
        tokenizer: Corresponding tokenizer
        id2label: Dictionary mapping label IDs to label strings

    Returns:
        List of (word, predicted_tag) tuples
    """
    # Tokenize the input
    words = sentence.split()
    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    model.to(device)
    model.eval()

    # Run inference (no gradient computation needed)
    with torch.no_grad():
        outputs = model(**inputs)

    # Get predicted label for each token
    predictions = torch.argmax(outputs.logits, dim=-1)[0].tolist()
    word_ids = inputs.word_ids()

    # Map predictions back to original words (skip subwords and special tokens)
    word_predictions = {}
    for token_idx, word_idx in enumerate(word_ids):
        if word_idx is None:
            continue
        if word_idx not in word_predictions:
            # Only take the prediction from the FIRST subword
            word_predictions[word_idx] = id2label[predictions[token_idx]]

    return [(words[i], word_predictions[i]) for i in sorted(word_predictions.keys())]


print("✅ Inference function defined.")

In [ ]:
# ─────────────────────────────────────────
# POS Tagging Inference on Custom Sentences
# ─────────────────────────────────────────
test_sentences = [
    "The quick brown fox jumps over the lazy dog",
    "Apple is planning to open a new store in Hyderabad next year",
    "She has been working on artificial intelligence research since 2020",
    "The President signed the new climate bill into law yesterday",
    "Deep learning models require large amounts of labeled training data"
]

print("🏷️  POS TAGGING PREDICTIONS")
print("=" * 70)

for sentence in test_sentences:
    print(f"\n📝 Sentence: '{sentence}'")
    pos_results = predict_tags(sentence, pos_model, tokenizer, pos_id2label)
    print("   Word                  → POS Tag")
    print("   " + "-" * 40)
    for word, tag in pos_results:
        print(f"   {word:<22} → {tag}")

In [ ]:
# ─────────────────────────────────────────
# Chunking Inference on Custom Sentences
# ─────────────────────────────────────────
print("📦 CHUNKING PREDICTIONS")
print("=" * 70)

for sentence in test_sentences[:3]:  # Show first 3 for brevity
    print(f"\n📝 Sentence: '{sentence}'")
    chunk_results = predict_tags(sentence, chunk_model, tokenizer, chunk_id2label)
    print("   Word                  → Chunk Tag")
    print("   " + "-" * 40)
    for word, tag in chunk_results:
        print(f"   {word:<22} → {tag}")

    # Group into chunks for visual readability
    print("\n   📦 Grouped Chunks:")
    current_chunk = []
    current_type = ""
    chunks_grouped = []
    for word, tag in chunk_results:
        if tag.startswith("B-"):
            if current_chunk:
                chunks_grouped.append((current_type, " ".join(current_chunk)))
            current_chunk = [word]
            current_type = tag[2:]  # Strip B-
        elif tag.startswith("I-"):
            current_chunk.append(word)
        else:  # O
            if current_chunk:
                chunks_grouped.append((current_type, " ".join(current_chunk)))
                current_chunk = []
                current_type = ""
            chunks_grouped.append(("O", word))
    if current_chunk:
        chunks_grouped.append((current_type, " ".join(current_chunk)))
    for chunk_type, chunk_text in chunks_grouped:
        marker = "🟢" if chunk_type == "NP" else "🔵" if chunk_type == "VP" else "⚪"
        print(f"   {marker} [{chunk_type}] {chunk_text}")

## 8. 📊 Results Visualization

In [ ]:
# ─────────────────────────────────────────
# Side-by-side Comparison of Both Tasks
# ─────────────────────────────────────────
metrics = ["Precision", "Recall", "F1 Score", "Accuracy"]

# Replace these with actual results after training
pos_scores  = [
    pos_test_results["eval_precision"],
    pos_test_results["eval_recall"],
    pos_test_results["eval_f1"],
    pos_test_results["eval_accuracy"]
]
chunk_scores = [
    chunk_test_results["eval_precision"],
    chunk_test_results["eval_recall"],
    chunk_test_results["eval_f1"],
    chunk_test_results["eval_accuracy"]
]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, pos_scores,   width, label="POS Tagging",  color="#4C72B0", alpha=0.85)
bars2 = ax.bar(x + width/2, chunk_scores, width, label="Chunking",     color="#DD8452", alpha=0.85)

ax.set_xlabel("Metric", fontsize=13)
ax.set_ylabel("Score", fontsize=13)
ax.set_title("DistilBERT Fine-tuning Results: POS Tagging vs Chunking", fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylim(0, 1.1)
ax.legend(fontsize=12)

for bar in bars1:
    ax.annotate(f"{bar.get_height():.3f}",
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=10)
for bar in bars2:
    ax.annotate(f"{bar.get_height():.3f}",
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig("results_comparison.png", dpi=150)
plt.show()
print("✅ Results comparison chart saved.")

In [ ]:
# ─────────────────────────────────────────
# Colored Token Annotation Visualization
# ─────────────────────────────────────────
def visualize_tags(sentence: str, task: str = "pos"):
    """
    Visualizes POS or Chunk tags above each word using matplotlib.
    """
    if task == "pos":
        tagged = predict_tags(sentence, pos_model, tokenizer, pos_id2label)
        title = "POS Tagging Visualization"
        # Color scheme by POS category
        color_map = {
            "NN": "#AED6F1", "NNS": "#AED6F1", "NNP": "#85C1E9", "NNPS": "#85C1E9",
            "VB": "#A9DFBF", "VBZ": "#A9DFBF", "VBD": "#A9DFBF", "VBG": "#A9DFBF",
            "JJ": "#FAD7A0", "JJR": "#FAD7A0", "JJS": "#FAD7A0",
            "DT": "#D7DBDD", "IN": "#EBDEF0", "CC": "#FDFEFE",
            "RB": "#FDEBD0", "PRP": "#D5F5E3"
        }
    else:
        tagged = predict_tags(sentence, chunk_model, tokenizer, chunk_id2label)
        title = "Chunking Visualization"
        color_map = {
            "B-NP": "#AED6F1", "I-NP": "#85C1E9",
            "B-VP": "#A9DFBF", "I-VP": "#7DCEA0",
            "B-PP": "#FAD7A0", "I-PP": "#F8C471",
            "B-ADJP": "#D7DBDD", "I-ADJP": "#BFC9CA",
            "O": "#FDFEFE"
        }

    fig, ax = plt.subplots(figsize=(max(14, len(tagged) * 1.2), 2.5))
    ax.axis("off")
    ax.set_title(f"{title}\n'{sentence}'", fontsize=11, fontweight="bold", pad=10)

    for i, (word, tag) in enumerate(tagged):
        x_pos = i * 1.1
        color = color_map.get(tag, "#FDFEFE")

        # Draw box for word
        rect = mpatches.FancyBboxPatch(
            (x_pos - 0.45, 0.15), 0.9, 0.45,
            boxstyle="round,pad=0.05",
            facecolor=color, edgecolor="gray", linewidth=0.8
        )
        ax.add_patch(rect)
        ax.text(x_pos, 0.38, word, ha="center", va="center", fontsize=9, fontweight="bold")
        ax.text(x_pos, 0.08, tag,  ha="center", va="center", fontsize=7.5, color="#2C3E50")

    ax.set_xlim(-0.6, len(tagged) * 1.1)
    ax.set_ylim(-0.05, 0.75)
    plt.tight_layout()
    fname = f"{task}_visualization.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved: {fname}")


# Visualize on a demo sentence
demo = "The quick brown fox jumps over the lazy dog"
visualize_tags(demo, task="pos")
visualize_tags(demo, task="chunk")

In [ ]:
# ─────────────────────────────────────────
# Print Final Summary Table
# ─────────────────────────────────────────
print("\n" + "=" * 60)
print("         FINAL RESULTS SUMMARY")
print("=" * 60)
print(f"{'Metric':<20} {'POS Tagging':>15} {'Chunking':>15}")
print("-" * 55)
print(f"{'Precision':<20} {pos_test_results['eval_precision']:>15.4f} {chunk_test_results['eval_precision']:>15.4f}")
print(f"{'Recall':<20} {pos_test_results['eval_recall']:>15.4f} {chunk_test_results['eval_recall']:>15.4f}")
print(f"{'F1 Score':<20} {pos_test_results['eval_f1']:>15.4f} {chunk_test_results['eval_f1']:>15.4f}")
print(f"{'Accuracy':<20} {pos_test_results['eval_accuracy']:>15.4f} {chunk_test_results['eval_accuracy']:>15.4f}")
print(f"{'Eval Loss':<20} {pos_test_results['eval_loss']:>15.4f} {chunk_test_results['eval_loss']:>15.4f}")
print("=" * 60)
print("\n✅ Both models fine-tuned and evaluated successfully!")

## 9. 🏁 Conclusion

### Summary

In this notebook, we successfully:

1. **Loaded & Explored** the CoNLL-2003 dataset — a standard NLP benchmark with POS and Chunk annotations.

2. **Tokenized** the data with DistilBERT's WordPiece tokenizer and carefully **aligned labels** using the `-100` masking strategy for subword tokens.

3. **Fine-tuned DistilBERT** (`distilbert-base-uncased`) for:
   - **POS Tagging** — predicting grammatical roles (NN, VBZ, DT, etc.) for each word
   - **Chunking** — identifying phrase boundaries (B-NP, I-VP, O, etc.)

4. **Evaluated** both models using **seqeval** metrics: Precision, Recall, F1, and Accuracy.

5. **Visualized** predictions on custom sentences with color-coded token annotations.

### Key Learnings

| Concept | Description |
|---|---|
| **Transformer Fine-tuning** | Pre-trained weights + task-specific head |
| **Token Classification** | Label each token instead of full sequence |
| **WordPiece Subword Tokenization** | Handles OOV words via subwords |
| **Label Alignment (-100)** | Mask continuation subwords in loss |
| **BIO Tagging** | B=Begin, I=Inside, O=Outside chunks |
| **seqeval Metrics** | Span-level evaluation for sequence labeling |
| **DataCollatorForTokenClassification** | Efficient padding for variable-length sequences |

### References
- [HuggingFace Transformers](https://huggingface.co/docs/transformers)
- [CoNLL-2003 Dataset](https://huggingface.co/datasets/conll2003)
- [DistilBERT Paper](https://arxiv.org/abs/1910.01108)
- [seqeval Library](https://github.com/chakki-works/seqeval)